# Chapter 1 ASIC Hard-Case Comparison Local Review

This notebook is the local-safe review layer for the saved ASIC 24h hard-case comparison package. It uses only the approved aggregate export bundle.

It prefers `cluster-results/chapter1_true_results/evaluation/asic/hard_cases/primary_medians/logistic_regression/asic_hard_case_comparison/` when that mirror is present locally and falls back to `artifacts/chapter1/evaluation/asic/hard_cases/primary_medians/logistic_regression/asic_hard_case_comparison/` otherwise.

This notebook does **not** read `stay_level_comparison_dataset.csv`, and it does **not** rebuild the comparison from restricted inputs.

Embedded outputs, if present, reflect the last time the notebook was executed and may predate the current artifact-source resolution rules.


## 1. Setup / Paths


In [ ]:
from __future__ import annotations

import os
import sys
from io import BytesIO
from pathlib import Path

import pandas as pd

if "MPLCONFIGDIR" not in os.environ:
    mplconfigdir = Path("/tmp") / "chapter1_mortality_decomposition_matplotlib"
    mplconfigdir.mkdir(parents=True, exist_ok=True)
    os.environ["MPLCONFIGDIR"] = str(mplconfigdir)

try:
    from IPython import get_ipython
    from IPython.display import Image, Markdown, display
    ipython_shell = get_ipython()
    if ipython_shell is None:
        raise RuntimeError("Plain Python execution: use fallback display helpers.")
    try:
        ipython_shell.run_line_magic("matplotlib", "inline")
    except Exception:
        pass
except Exception:
    ipython_shell = None

    class Markdown(str):
        pass

    class Image:
        def __init__(self, data=None, filename=None):
            self.data = data
            self.filename = filename

        def __repr__(self) -> str:
            return f"Image(filename={self.filename!r})"

    def display(obj: object) -> None:
        print(obj)

import matplotlib
if ipython_shell is None:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt


def finalize_figure(fig=None) -> None:
    fig = fig or plt.gcf()
    backend = matplotlib.get_backend().lower()
    if "agg" in backend:
        if ipython_shell is not None:
            buffer = BytesIO()
            fig.savefig(buffer, format="png", bbox_inches="tight")
            buffer.seek(0)
            display(Image(data=buffer.getvalue()))
            buffer.close()
        plt.close(fig)
        return
    plt.show()


def looks_like_repo_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "chapter1_mortality_decomposition").exists()


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    search_roots = [start, *start.parents]
    for candidate in search_roots:
        if looks_like_repo_root(candidate):
            return candidate
    for anchor in search_roots:
        try:
            child_dirs = [child for child in anchor.iterdir() if child.is_dir()]
        except Exception:
            continue
        for child in child_dirs:
            if looks_like_repo_root(child):
                return child
    return start


WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = find_repo_root(WORKING_DIRECTORY)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from chapter1_mortality_decomposition.notebook_artifact_sources import (
    build_artifact_source_markdown,
    resolve_notebook_artifact_source,
)

COMPARISON_SOURCE = resolve_notebook_artifact_source(
    Path("evaluation") / "asic" / "hard_cases" / "primary_medians" / "logistic_regression" / "asic_hard_case_comparison",
    label="ASIC hard-case comparison aggregate package",
    repo_root=REPO_ROOT,
)
COMPARISON_ROOT = COMPARISON_SOURCE.path
COMPARISON_TABLE_PATH = COMPARISON_ROOT / "comparison_table.csv"
EFFECT_SIZE_PLOT_DATA_PATH = COMPARISON_ROOT / "effect_size_plot_data.csv"
SUMMARY_PATH = COMPARISON_ROOT / "summary.md"
FIGURE_PATH = COMPARISON_ROOT / "effect_size_figure.png"
EARLY_LATE_ROOT = COMPARISON_ROOT / "early_vs_late_death_split"
EARLY_LATE_SUMMARY_PATH = EARLY_LATE_ROOT / "early_vs_late_fatal_timing_summary.csv"
EARLY_LATE_FIGURE_PATH = EARLY_LATE_ROOT / "early_vs_late_low_pred_share.png"
EARLY_LATE_NOTE_PATH = EARLY_LATE_ROOT / "early_vs_late_interpretation_note.md"

REQUIRED_PATHS = [
    COMPARISON_TABLE_PATH,
    EFFECT_SIZE_PLOT_DATA_PATH,
    SUMMARY_PATH,
    FIGURE_PATH,
    EARLY_LATE_SUMMARY_PATH,
    EARLY_LATE_FIGURE_PATH,
    EARLY_LATE_NOTE_PATH,
]

missing_paths = [path for path in REQUIRED_PATHS if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "The approved hard-case comparison aggregate bundle is incomplete. Missing: "
        + ", ".join(str(path) for path in missing_paths)
    )

comparison_table = pd.read_csv(COMPARISON_TABLE_PATH)
effect_size_plot_data = pd.read_csv(EFFECT_SIZE_PLOT_DATA_PATH)
early_vs_late_summary = pd.read_csv(EARLY_LATE_SUMMARY_PATH)
summary_text = SUMMARY_PATH.read_text(encoding="utf-8")
early_vs_late_note = EARLY_LATE_NOTE_PATH.read_text(encoding="utf-8")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


def relative_to_repo(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(REPO_ROOT.resolve()))
    except ValueError:
        return str(path.resolve())


def render_image(path: Path, title: str) -> None:
    if not path.exists():
        display(Markdown(f"_Missing figure: `{relative_to_repo(path)}`_"))
        return
    image = plt.imread(path)
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.imshow(image)
    ax.axis("off")
    ax.set_title(title)
    fig.tight_layout()
    finalize_figure(fig)


display(Markdown(f"Working directory: `{WORKING_DIRECTORY}`"))
display(Markdown(f"Repository root: `{REPO_ROOT}`"))
display(Markdown(build_artifact_source_markdown([COMPARISON_SOURCE], display_root=REPO_ROOT)))
display(Markdown("This notebook consumes only the approved aggregate export bundle for local review."))

artifact_paths = pd.DataFrame(
    [
        {"artifact_name": "comparison table", "path": relative_to_repo(COMPARISON_TABLE_PATH), "exists": COMPARISON_TABLE_PATH.exists()},
        {"artifact_name": "effect-size plot data", "path": relative_to_repo(EFFECT_SIZE_PLOT_DATA_PATH), "exists": EFFECT_SIZE_PLOT_DATA_PATH.exists()},
        {"artifact_name": "summary note", "path": relative_to_repo(SUMMARY_PATH), "exists": SUMMARY_PATH.exists()},
        {"artifact_name": "effect-size figure", "path": relative_to_repo(FIGURE_PATH), "exists": FIGURE_PATH.exists()},
        {"artifact_name": "early-vs-late timing summary", "path": relative_to_repo(EARLY_LATE_SUMMARY_PATH), "exists": EARLY_LATE_SUMMARY_PATH.exists()},
        {"artifact_name": "early-vs-late timing figure", "path": relative_to_repo(EARLY_LATE_FIGURE_PATH), "exists": EARLY_LATE_FIGURE_PATH.exists()},
        {"artifact_name": "early-vs-late interpretation note", "path": relative_to_repo(EARLY_LATE_NOTE_PATH), "exists": EARLY_LATE_NOTE_PATH.exists()},
    ]
)
display(Markdown("### Aggregate Artifact Paths"))
display(artifact_paths)


## 2. Summary Note


In [ ]:
display(Markdown(summary_text))


## 3. Main Comparison Table


In [ ]:
display(Markdown("### Main Table"))
display(comparison_table)

display(Markdown("### Strongest absolute effect sizes"))
display(effect_size_plot_data.loc[:, [
    "figure_label",
    "effect_size_type",
    "effect_size_basis",
    "standardized_difference",
    "absolute_standardized_difference",
]].head(5))


## 4. Effect-Size Figure


In [ ]:
render_image(FIGURE_PATH, "ASIC 24h hard-case comparison effect sizes")


## 5. Early-vs-Late Fatal Timing Appendix


In [ ]:
display(Markdown("### Early-vs-Late Timing Summary"))
display(early_vs_late_summary)

display(Markdown("### Early-vs-Late Interpretation Note"))
display(Markdown(early_vs_late_note))

display(Markdown("### Early-vs-Late Figure"))
render_image(EARLY_LATE_FIGURE_PATH, "Early vs late ICU death split")
